In [ ]:
# 📜 License & Data Source Attribution

## Code License
#This notebook's code is published as open-source software under the **Apache License 2.0**.

## Data Source & Reuse Policy
#* **Data Provider:** [Eurostat](https://ec.europa.eu/eurostat) (European Commission)
#* **Datasets Used:** 
#  * `tour_ce_omd` / `tour_occ_nim` (Nights spent at tourist accommodation establishments)
#  * `prc_hicp_midx` (HICP - Actual rentals for housing, COICOP `CP0411`)
#* **Legal Reuse Notice:** Data reused under the European Commission open data policy ([Commission Decision 2011/833/EU](https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32011D0833)).

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# pip install eurostatapiclient pandas matplotlib
import gzip
import io
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import FileLink

# ---------------------------------------------------------
# 1. API DATA FETCH FUNCTION
# ---------------------------------------------------------
def fetch_eurostat_safe(dataset_code, fallback_code=None):
    """Downloads raw datasets from Eurostat API SDMX 2.1 with automatic fallback."""
    url = f"https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset_code}?format=TSV&compressed=true"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200 and fallback_code:
        print(f"Warning: {dataset_code} restricted. Switching to fallback dataset: {fallback_code}...")
        url = f"https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{fallback_code}?format=TSV&compressed=true"
        response = requests.get(url, headers=headers)
        
    if response.status_code != 200:
        raise ValueError(f"Download failed with status {response.status_code}: {response.text[:150]}")

    with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as gz:
        df = pd.read_csv(gz, sep='\t', low_memory=False)
        
    first_col = df.columns[0]
    dim_names = [d.strip() for d in first_col.split('\\')[0].split(',')]
    split_dims = df[first_col].str.split(',', expand=True)
    
    for idx, name in enumerate(dim_names):
        df[name] = split_dims[idx].str.strip()
        
    df = df.drop(columns=[first_col])
    df.columns = [c.strip() for c in df.columns]
    return df


# ---------------------------------------------------------
# 2. FETCH RAW DATASETS
# ---------------------------------------------------------
print("Downloading Tourism Data...")
df_str = fetch_eurostat_safe('tour_ce_omd', fallback_code='tour_occ_nim')

print("Downloading Rent Data...")
df_rent = fetch_eurostat_safe('prc_hicp_midx')

print("\n--- Download Complete ---")
print("Tourism Data Shape:", df_str.shape)
print("Rent Data Shape:", df_rent.shape)


# ---------------------------------------------------------
# 3. CLEAN & PROCESS DATASETS
# ---------------------------------------------------------
# Tourism Data
df_tour = df_str.copy()
time_cols_tour = [c for c in df_tour.columns if 'M' in c or (c.startswith('20') and len(c) == 7)]

df_tour_long = df_tour.melt(
    id_vars=['geo'],
    value_vars=time_cols_tour,
    var_name='period',
    value_name='tourist_nights'
)
df_tour_long['tourist_nights'] = pd.to_numeric(
    df_tour_long['tourist_nights'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)
df_tour_long['date'] = pd.to_datetime(df_tour_long['period'].str.replace('M', '-'), format='%Y-%m', errors='coerce')
df_tour_cleaned = df_tour_long.dropna(subset=['tourist_nights', 'date']).copy()

# Rent Data
rent_housing = df_rent[df_rent['coicop'].isin(['CP0411', '0411'])].copy()
time_cols_rent = [c for c in rent_housing.columns if 'M' in c or (c.startswith('20') and len(c) == 7)]

df_rent_long = rent_housing.melt(
    id_vars=['geo', 'coicop', 'unit'],
    value_vars=time_cols_rent,
    var_name='period',
    value_name='rent_index'
)
df_rent_long['rent_index'] = pd.to_numeric(
    df_rent_long['rent_index'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)
df_rent_long['date'] = pd.to_datetime(df_rent_long['period'].str.replace('M', '-'), format='%Y-%m', errors='coerce')
df_rent_cleaned = df_rent_long.dropna(subset=['rent_index', 'date']).copy()


# ---------------------------------------------------------
# 4. ANALYSIS & PLOTTING FUNCTIONS
# ---------------------------------------------------------
def analyze_country(country_code, country_name):
    # Filter and Aggregate
    t_c = df_tour_cleaned[df_tour_cleaned['geo'] == country_code].groupby('date')['tourist_nights'].sum().reset_index()
    r_c = df_rent_cleaned[df_rent_cleaned['geo'] == country_code].groupby('date')['rent_index'].mean().reset_index()
    
    merged = pd.merge(t_c, r_c, on='date', how='inner').sort_values('date').reset_index(drop=True)
    
    if merged.empty:
        print(f"No overlapping data found for {country_name} ({country_code}).")
        return
    
    # 1. Plot Raw Levels Comparison
    corr = merged['tourist_nights'].corr(merged['rent_index'])
    fig, ax1 = plt.subplots(figsize=(12, 5))

    color = 'tab:blue'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Tourist Nights Spent', color=color)
    ax1.plot(merged['date'], merged['tourist_nights'], color=color, linewidth=1.5, label='Tourist Nights')
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('Actual Housing Rent Index (2015=100)', color=color)
    ax2.plot(merged['date'], merged['rent_index'], color=color, linewidth=2.5, linestyle='--', label='Rent Index')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f'Tourism Activity vs. Housing Rent Index in {country_name}\nPearson Correlation: {corr:.2f}', fontsize=13)
    fig.tight_layout()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2. Plot Year-over-Year (YoY) Growth (De-seasonalized)
    merged['rent_yoy'] = merged['rent_index'].pct_change(12) * 100
    merged['tour_yoy'] = merged['tourist_nights'].pct_change(12) * 100
    merged_yoy = merged.dropna(subset=['rent_yoy', 'tour_yoy']).copy()

    corr_yoy = merged_yoy['tour_yoy'].corr(merged_yoy['rent_yoy'])

    plt.figure(figsize=(12, 5))
    plt.plot(merged_yoy['date'], merged_yoy['rent_yoy'], label='Rent Index YoY Growth (%)', color='tab:red', linewidth=2.5)
    plt.plot(merged_yoy['date'], merged_yoy['tour_yoy'], label='Tourism Nights YoY Growth (%)', color='tab:blue', alpha=0.6, linewidth=1.5)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5)
    plt.title(f'Year-over-Year (YoY) Growth in {country_name} (Seasonality Removed)\nYoY Correlation: {corr_yoy:.2f}', fontsize=13)
    plt.xlabel('Date')
    plt.ylabel('Percentage Change YoY (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Run combined analysis for Spain (ES) and Italy (IT)
analyze_country('ES', 'Spain')
analyze_country('IT', 'Italy')


# ---------------------------------------------------------
# 5. EXPORT CLEANED CSV FILES
# ---------------------------------------------------------
df_rent_cleaned.to_csv('cleaned_rent_data.csv', index=False)
df_tour_cleaned.to_csv('cleaned_tourism_data.csv', index=False)

print("Files saved successfully to Kaggle working directory!")

# Generate clickable download links in Kaggle notebook
display(FileLink('cleaned_rent_data.csv'))
display(FileLink('cleaned_tourism_data.csv'))